In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import numpy.typing as npt

from climate_attitudes.visualisation import configure_mpl
from ising import Ising

configure_mpl(Path("../fonts/"))

np.set_printoptions(linewidth=200)

RANDOM_SEED = 202607161539
rng = np.random.default_rng(RANDOM_SEED)

In [ ]:
def get_transition_matrix(model: Ising) -> npt.NDArray[np.float64]:
    N = model.size
    Y0 = (2 * ((np.arange(1 << N)[:, None] >> np.arange(N)) & 1) - 1).astype(np.float64)
    X = np.ones(Y0.shape[0])

    heff = model.parallel_glauber_theta_batch(Y0, X, model.h, model.j, model.adj)
    sum_log2cosh_heff = np.log(2 * np.cosh(heff)).sum(axis=-1)
    s_dot_heff = heff @ Y0.T

    p_transition = np.exp(s_dot_heff - sum_log2cosh_heff[:, None]).T

    return p_transition

In [ ]:
def get_all_transition_matrices(
    params: npt.NDArray[np.float64],
    int_amount: float,
    int_idx: int,
) -> npt.NDArray[np.float64]:
    N = 8
    R = np.empty((params.shape[0], 2**N, 2**N), dtype=np.float64)
    for repeat in range(params.shape[0]):
        h, j = Ising.unpack_params(params[repeat], k=0)
        h[int_idx] += int_amount
        model = Ising(
            field=h,
            coupling=j,
            infer_structure=True,
            rng=np.random.default_rng(RANDOM_SEED),
        )
        R[repeat] = get_transition_matrix(model)
    return R

In [ ]:
def avg_value(
    p: npt.NDArray[np.float64],
    all_states: npt.NDArray[np.int64],
    idx: int,
) -> float:
    return (p[:, None] * all_states)[:, idx].sum()

In [ ]:
null_measurements = np.load(
    "../reports/thesis/results/data/model/all_interventions_activation_probability/ising_00_5.npz"
)["p"]
int_measurements = np.load(
    "../reports/thesis/results/data/model/all_interventions_activation_probability/ising_25_5.npz"
)["p"]

In [ ]:
Y0 = np.load("../reports/thesis/results/data/model/all_interventions/ising_25.npz")[
    "Y0"
][:, 1, :]

In [ ]:
np.load("../reports/thesis/results/data/model/all_interventions/ising_25.npz")[
    "Y0"
].shape

In [ ]:
null_measurements.shape

In [ ]:
effect = (int_measurements[:, :, 5, 7] - null_measurements[:, :, 5, 7]).mean(axis=0)
effect.shape

In [ ]:
effect[:10]

In [ ]:
Y0.shape

In [ ]:
effect[(Y0[:, 2] < 0) & (Y0[:, 5] < 0)].mean()

In [ ]:
effect[(Y0[:, 2] > 0) & (Y0[:, 5] < 0)].mean()

In [ ]:
effect[(Y0[:, 5] < 0)].mean()

In [ ]:
effect[(Y0[:, 2] < 0)].mean()

In [ ]:
threshold = np.percentile(effect, 75)

In [ ]:
high_effect = effect >= threshold

In [ ]:
Y0[high_effect & (Y0[:, 2] < 0) & (Y0[:, 1] < 0)].shape[0] / Y0[high_effect].shape[0]

In [ ]:
Y0[high_effect & (Y0[:, 2] < 0) & (Y0[:, 0] < 0)].shape[0] / Y0[high_effect].shape[0]

In [ ]:
Y0[~high_effect & (Y0[:, 2] < 0) & (Y0[:, 1] < 0)].shape[0] / Y0[~high_effect].shape[0]

In [ ]:
Y0[~high_effect & (Y0[:, 2] < 0) & (Y0[:, 0] < 0)].shape[0] / Y0[~high_effect].shape[0]

In [ ]:
Y0[high_effect & (Y0[:, 2] < 0) & (Y0[:, 1] < 0)].mean(axis=0)

In [ ]:
Y0[~high_effect & (Y0[:, 2] < 0) & (Y0[:, 1] < 0)].mean(axis=0)

In [ ]:
Y0[high_effect & (Y0[:, 2] < 0) & (Y0[:, 1] > 0)].mean(axis=0)

In [ ]:
Y0[~high_effect & (Y0[:, 2] < 0) & (Y0[:, 1] > 0)].mean(axis=0)

In [ ]:
np.corrcoef(Y0[:, 0], Y0[:, 1])

In [ ]:
params = np.load("../reports/thesis/results/data/model/all_interventions/ising_25.npz")[
    "params"
]
R_null = get_all_transition_matrices(params, 0, 5)
R_int = get_all_transition_matrices(params, 2.5, 0)
R = R_int - R_null

In [ ]:
labels = [
    "CC Real",
    "CC Human",
    "CC Worry",
    "CC Others Worry",
    "Weather Worry",
    "Politics",
    "CC Impact",
    "CC Action",
]

In [ ]:
N = 8
s0 = np.array([0, -1, -1, 0, 0, 0, 0, 0])

fixed_spins = s0 != 0
n_fixed_spins = fixed_spins.sum()
p_uniform = 1 / 2 ** (N - n_fixed_spins)

Y0 = (2 * ((np.arange(1 << N)[:, None] >> np.arange(N)) & 1) - 1).astype(np.float64)
p0 = np.zeros(2**N)

for s_i, s in enumerate(Y0):
    if (s[fixed_spins] != s0[fixed_spins]).any():
        continue
    p0[s_i] = p_uniform

p_null = p0
p_int = p0

avg_vals = np.empty((6, 8), dtype=np.float64)
for i in range(8):
    avg_vals[0, i] = avg_value(p_int, Y0, i)  # - avg_value(p_null, Y0, i)

for t in range(1, 6):
    p_null = R_null[0] @ p_null
    p_int = R_int[0] @ p_int
    for i in range(8):
        avg_vals[t, i] = avg_value(p_int, Y0, i)  # - avg_value(p_null, Y0, i)


fig, ax = plt.subplots(figsize=(5.77, 3), constrained_layout=True)
for i in range(8):
    ax.plot(np.arange(6), avg_vals[:, i], "o-", markerfacecolor=None, label=labels[i])
ax.legend(bbox_to_anchor=(0.5, 1.05), loc="lower center", ncols=4, fontsize=7)
ax.grid(visible=True)

In [ ]:
N = 8
s0 = np.array([0, 1, -1, 0, 0, 0, 0, 0])

fixed_spins = s0 != 0
n_fixed_spins = fixed_spins.sum()
p_uniform = 1 / 2 ** (N - n_fixed_spins)

Y0 = (2 * ((np.arange(1 << N)[:, None] >> np.arange(N)) & 1) - 1).astype(np.float64)
p0 = np.zeros(2**N)

for s_i, s in enumerate(Y0):
    if (s[fixed_spins] != s0[fixed_spins]).any():
        continue
    p0[s_i] = p_uniform

p_null = p0
p_int = p0

avg_vals = np.empty((6, 8), dtype=np.float64)
for i in range(8):
    avg_vals[0, i] = avg_value(p_int, Y0, i)  # - avg_value(p_null, Y0, i)

for t in range(1, 6):
    p_null = R_null[0] @ p_null
    p_int = R_int[0] @ p_int
    for i in range(8):
        avg_vals[t, i] = avg_value(p_int, Y0, i)  # - avg_value(p_null, Y0, i)


fig, ax = plt.subplots(figsize=(5.77, 3), constrained_layout=True)
for i in range(8):
    ax.plot(np.arange(6), avg_vals[:, i], "o-", markerfacecolor=None, label=labels[i])
ax.legend(bbox_to_anchor=(0.5, 1.05), loc="lower center", ncols=4, fontsize=7)
ax.grid(visible=True)

In [ ]:
N = 8
s0 = np.array([0, 1, -1, 0, 0, 0, 0, -1])

fixed_spins = s0 != 0
n_fixed_spins = fixed_spins.sum()
p_uniform = 1 / 2 ** (N - n_fixed_spins)

Y0 = (2 * ((np.arange(1 << N)[:, None] >> np.arange(N)) & 1) - 1).astype(np.float64)
p0 = np.zeros(2**N)

for s_i, s in enumerate(Y0):
    if (s[fixed_spins] != s0[fixed_spins]).any():
        continue
    p0[s_i] = p_uniform

p_null = p0
p_int = p0

avg_vals = np.empty((6, 8), dtype=np.float64)
for i in range(8):
    avg_vals[0, i] = avg_value(p_int, Y0, i)  # - avg_value(p_null, Y0, i)

for t in range(1, 6):
    p_null = R_null[0] @ p_null
    p_int = R_int[0] @ p_int
    for i in range(8):
        avg_vals[t, i] = avg_value(p_int, Y0, i)  # - avg_value(p_null, Y0, i)


fig, ax = plt.subplots(figsize=(5.77, 3), constrained_layout=True)
for i in range(8):
    ax.plot(np.arange(6), avg_vals[:, i], "o-", markerfacecolor=None, label=labels[i])
ax.legend(bbox_to_anchor=(0.5, 1.05), loc="lower center", ncols=4, fontsize=7)
ax.grid(visible=True)